# quadrucat: training on a Kaggle GPU (baseline and cat mode)

**Launched automatically** by `scripts/kaggle_run.py` (it rewrites the first code cell, see README).
The steps below are only for running it by hand.

**Before running:** Settings (right panel) → *Accelerator*: **GPU T4 x1** or **P100**, *Internet*: **On**
(needs a phone-verified Kaggle account). Add the dataset with `quadrucat.zip` via *Add Input*.

Best way to run the long training: **Save Version → Save & Run All (Commit)**. It runs in the
background (up to 12 h) even if you close the browser; results appear in the *Output* tab.
Set `QUICK = True` first to check everything in ~15 min.

**Choose what to train with `TASK`** in the first code cell:
* `base`: velocity-tracking baseline (`configs/base.yaml`), ~20 min.
* `cat_walk`: cat-like walk with direct register (`configs/cat_walk.yaml`), same cost as the baseline.
* `cat_fall`: land on the feet after a drop (`configs/cat_fall.yaml`, 100M steps, full-body collisions: slower per step).
* `cat_both`: `cat_walk` then `cat_fall` in one commit.

Extra settings go in `EXTRA_SET`, e.g. `"env.overrides.reward_config.scales.direct_register=10.0"`.
See EXPERIMENTS.md §5-6 for what each knob does.

In [ ]:
# --- parameters (scripts/kaggle_run.py replaces the values of these lines) ---
QUICK = False          # True: smoke tests only (no long training)
CONFIG_FILES = []      # explicit list of configs, e.g. ["configs/cat_walk_v2.yaml"]; overrides TASK
EXPECTED_CODE = ""     # code version that the dataset must contain (CODE_VERSION.txt); "" = do not check
TASK = "cat_walk"      # "base" | "cat_walk" | "cat_fall" | "cat_both"
SEED = 0
NUM_ENVS = None        # e.g. 4096 if the GPU runs out of memory (8192 is the base value)
EXTRA_SET = ""         # extra --set overrides, space separated, e.g. "ppo.num_timesteps=100_000_000"
NOTES = ""             # your hypothesis for this run (saved in config.yaml)
# --- end parameters ---
CONFIGS = CONFIG_FILES or {"base": ["configs/base.yaml"], "cat_walk": ["configs/cat_walk.yaml"],
           "cat_fall": ["configs/cat_fall.yaml"], "cat_both": ["configs/cat_walk.yaml", "configs/cat_fall.yaml"]}[TASK]

import subprocess, sys
FAILED = []            # commands that failed; reported (and raised) in the last cell
def run(cmd, log=None, max_repeats=3):
    """Run a shell command, stream its output, and hide floods of repeated GPU warnings
    (e.g. 'CCD overflow', which can freeze the browser tab). Optionally tee to a log file."""
    seen = {}
    f = open(log, 'a') if log else None
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        key = line.split(' to ')[0] if 'please increase' in line else None
        if f: f.write(line)
        if key:
            seen[key] = seen.get(key, 0) + 1
            if seen[key] > max_repeats:
                if seen[key] == max_repeats + 1:
                    print(f'[hidden: further "{key.strip()}" lines]', flush=True)
                continue
        print(line, end='', flush=True)
    p.wait()
    if f: f.close()
    if seen:
        print(f'WARNING: {sum(seen.values())} buffer-overflow lines (contacts dropped). Raise env.overrides.naccdmax.')
    if p.returncode:
        FAILED.append(cmd); print(f'!!! exit code {p.returncode}: {cmd}', flush=True)
    return p.returncode

!nvidia-smi

In [ ]:
import glob, os, shutil, subprocess, zipfile
# Locate the uploaded code. Kaggle usually auto-extracts a dataset zip, and the mount path
# varies (/kaggle/input/<dataset>/..., /kaggle/input/datasets/<user>/<dataset>/...).
print("Contents of /kaggle/input (first 40 entries):")
for i, p in enumerate(sorted(glob.glob('/kaggle/input/**', recursive=True))[:40]):
    print("  ", p)

def find_repo():
    for root in ['/kaggle/input', '/kaggle/working', '/kaggle']:
        hits = [h for h in glob.glob(f'{root}/**/pyproject.toml', recursive=True)
                if os.path.exists(os.path.join(os.path.dirname(h), 'scripts', 'train.py'))
                and '/kaggle/working/quadrucat' not in h]
        if hits:
            return os.path.dirname(sorted(hits, key=len)[0])
    for root in ['/kaggle/input', '/kaggle/working']:
        zips = glob.glob(f'{root}/**/*.zip', recursive=True)
        for z in zips:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith('pyproject.toml') for n in zf.namelist()):
                    zf.extractall('/kaggle/tmp_src')
                    return os.path.dirname(glob.glob('/kaggle/tmp_src/**/pyproject.toml', recursive=True)[0])
    return None

src = find_repo()
if src is None:
    raise SystemExit(
        "Could not find the quadrucat code. In the right panel click 'Add Input' -> 'Your Work' / "
        "'Datasets' and add the dataset where you uploaded quadrucat.zip (or upload quadrucat.zip to "
        "/kaggle/working with the 'Upload' button), then run this cell again.")
print("Found code in:", src)
version_file = os.path.join(src, 'CODE_VERSION.txt')
version = open(version_file).read().strip() if os.path.exists(version_file) else ''
print("Code version:", version or '(none)')
if EXPECTED_CODE and version.split()[:1] != [EXPECTED_CODE]:
    raise RuntimeError(f"The dataset holds code version {version!r}, expected {EXPECTED_CODE!r}: "
                       "Kaggle mounted an old dataset version. Launch again in a minute.")
REPO = '/kaggle/working/quadrucat'
shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); print(os.getcwd()); print(sorted(os.listdir()))

In [ ]:
# Pinned stack (MuJoCo Playground 0.2.0 = last version with the JAX Go1 tasks). ~3-5 min.
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!apt-get -qq install -y ffmpeg libegl1 > /dev/null 2>&1 || true

In [ ]:
# Must list a CUDA device. First call also downloads MuJoCo Menagerie (needs Internet On).
run('python scripts/check_install.py')
if FAILED: raise RuntimeError('broken install or no GPU')
run('python -m pytest -q')

In [ ]:
# Quick pipeline check on the GPU (a few minutes per task).
import yaml
ENVS = {yaml.safe_load(open(p)).get("env", {}).get("name", "") for p in CONFIGS} if CONFIG_FILES else set()
if CONFIG_FILES: TASK = "cat_fall" if ENVS == {"Go1CatFall"} else "cat_walk"
if TASK == "base":
    run('python scripts/train.py --config configs/smoke.yaml --notes "smoke on Kaggle GPU"')
if TASK in ("cat_walk", "cat_both"):
    run('python scripts/train.py --config configs/smoke_cat.yaml --notes "cat walk smoke on Kaggle GPU"')
if TASK in ("cat_fall", "cat_both"):
    run('python scripts/train.py --config configs/cat_fall.yaml --name smoke_catfall '
        '--set ppo.num_timesteps=2_000_000 ppo.num_evals=3 ppo.num_envs=1024 eval.num_episodes=16')

In [ ]:
# Full training for each config of TASK. Run dirs are collected in TRAINED.
import time
TRAINED = []
extra = (f"ppo.num_envs={NUM_ENVS} " if NUM_ENVS else "") + EXTRA_SET
if not QUICK:
    for cfg_path in CONFIGS:
        tag = os.path.basename(cfg_path).replace('.yaml', '')
        t_start = time.time()
        set_args = f"--set {extra}" if extra.strip() else ""
        rc = run(f'python scripts/train.py --config {cfg_path} --seed {SEED} {set_args} '
            f'--notes "{NOTES or tag} seed {SEED}, Kaggle GPU"', log=f'runs_train_{tag}_seed{SEED}.log')
        new = [r for r in glob.glob('runs/*') if os.path.getmtime(r) >= t_start and not os.path.basename(r).startswith(('smoke', 'passive'))]
        if rc == 0: TRAINED += sorted(new, key=os.path.getmtime)[-1:]
else:
    TRAINED = sorted([r for r in glob.glob('runs/*') if 'smoke' in os.path.basename(r)], key=os.path.getmtime)
print("runs to evaluate:", TRAINED)

In [ ]:
# Evaluate every run: metrics, robustness, cat metrics, video, export, sim2sim, figures.
os.environ['MUJOCO_GL'] = 'egl'
import yaml
# 'default' and 'noise2x' are always WITHOUT pushes, also for runs trained with pushes, so runs are comparable.
NOPUSH, PUSH = 'env.overrides.pert_config.enable=false', 'env.overrides.pert_config.enable=true'
for rdir in TRAINED:
    env_name = yaml.safe_load(open(f'{rdir}/config.yaml'))['env']['name']
    fig_dir = f'figures/{os.path.basename(rdir)}'
    print('=' * 30, rdir, env_name)
    if env_name == 'Go1CatFall':
        run('python scripts/evaluate_cat.py --config configs/cat_fall.yaml --zero-policy --tag passive')
        run(f'python scripts/evaluate_cat.py {rdir} --video || MUJOCO_GL=osmesa python scripts/evaluate_cat.py {rdir} --video')
        run(f'python scripts/evaluate_cat.py {rdir} --tag upside_down --set env.overrides.min_tilt=2.36')
        run(f'python scripts/evaluate_cat.py {rdir} --tag high --set "env.overrides.drop_height=[1.5,2.0]"')
        run(f'python scripts/plot_results.py {rdir} runs/passive_* --out-dir {fig_dir} --metric eval/episode_landed_on_feet')
    else:
        run(f'python scripts/evaluate.py {rdir} --video --set {NOPUSH} || MUJOCO_GL=osmesa python scripts/evaluate.py {rdir} --video --set {NOPUSH}')
        run(f'python scripts/evaluate.py {rdir} --tag pushes --set {PUSH}')
        run(f'python scripts/evaluate.py {rdir} --tag noise2x --set {NOPUSH} env.overrides.noise_config.level=2.0')
        run(f'python scripts/evaluate_cat.py {rdir} --video --set {NOPUSH} || MUJOCO_GL=osmesa python scripts/evaluate_cat.py {rdir} --video --set {NOPUSH}')
        run(f'python scripts/evaluate_cat.py {rdir} --tag pushes --set {PUSH}')
        run(f'python scripts/export_policy.py {rdir}')
        run(f'python scripts/sim2sim_mujoco.py {rdir}')
        run(f'python scripts/plot_results.py {rdir} --out-dir {fig_dir}')

In [ ]:
# Show the diagnostics and package everything for download (Output tab -> quadrucat_results.zip).
from IPython.display import Image, Video, display
for rdir in TRAINED:
    fig_dir = f'figures/{os.path.basename(rdir)}'
    for png in sorted(glob.glob(f'{fig_dir}/*.png')):
        print(png); display(Image(png))
    vids = sorted(glob.glob(f'{rdir}/eval_cat/*.mp4')) + sorted(glob.glob(f'{rdir}/eval/*.mp4'))
    if vids: print(vids[0]); display(Video(vids[0], embed=True, width=640))
with zipfile.ZipFile('/kaggle/working/quadrucat_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['runs', 'figures']:
        for p in glob.glob(f'{folder}/**/*', recursive=True):
            if os.path.isfile(p) and '/checkpoints/' not in p:
                zf.write(p)
    for p in glob.glob('*.log'): zf.write(p)
    zf.writestr('failures.txt', '\n'.join(FAILED))
print('-> /kaggle/working/quadrucat_results.zip')
if not TRAINED:
    raise RuntimeError("No run finished. Failed commands:\n" + "\n".join(FAILED))
